# Indoor Scene Change Detection — Training & Evaluation Pipeline


## 1. Environment Setup


In [ ]:
!nvidia-smi


In [ ]:
!pip install ultralytics pandas numpy scipy scikit-learn matplotlib seaborn opencv-python -q
import os, json, shutil, glob, zipfile, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from PIL import Image as PILImage
from sklearn.metrics import classification_report, confusion_matrix
from scipy.optimize import linear_sum_assignment
from ultralytics import YOLO
from google.colab import files
from IPython.display import Image, display

print("Libraries ready.")


## 5. Train YOLO11s


In [ ]:
EPOCHS = 60
IMGSZ = 640
!yolo detect train data=/content/data.yaml model=yolo11s.pt epochs={EPOCHS} imgsz={IMGSZ} \
    project=/content/runs name=yolo11s


In [ ]:
YOLO_WEIGHTS = '/content/runs/yolo11s/weights/best.pt'
assert os.path.exists(YOLO_WEIGHTS), "YOLO11s training did not produce best.pt — check the training log above."
print("YOLO11s weights:", YOLO_WEIGHTS)


### 5b. YOLO11s Training Curves


In [ ]:
yolo_results_csv = '/content/runs/yolo11s/results.csv'
if os.path.exists(yolo_results_csv):
    yolo_hist = pd.read_csv(yolo_results_csv)
    yolo_hist.columns = [c.strip() for c in yolo_hist.columns]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    axes[0].plot(yolo_hist['epoch'], yolo_hist['train/box_loss'], label='train/box_loss')
    axes[0].plot(yolo_hist['epoch'], yolo_hist['val/box_loss'], label='val/box_loss')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].set_title('YOLO11s -- Box Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(yolo_hist['epoch'], yolo_hist['metrics/mAP50(B)'], label='mAP50')
    axes[1].plot(yolo_hist['epoch'], yolo_hist['metrics/mAP50-95(B)'], label='mAP50-95')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('mAP')
    axes[1].set_title('YOLO11s -- Validation mAP'); axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('/content/yolo11s_training_curves.png', dpi=150)
    plt.show()
else:
    print(f"Couldn't find {yolo_results_csv} -- skipping training curves "
          f"(re-run Section 5's training cell first).")
